In [1]:
import numpy as np

from training.gridsearch import run_grid

from nn.model import Model
from nn.layers import Dense, xavier_uniform
from nn.activations import Tanh, Sigmoid, ReLU
from nn.dropout import Dropout
from nn.losses import BinaryCrossEntropy
from nn.optim import SGD
from nn.metrics import Accuracy
from nn.regularizers import L2
from nn.callbacks import EarlyStopping
from data_handler.data_loader import load_monk
from data_handler.data_splitter import split_train_val
from training.trainer import Trainer


# --------- 1) Functions required by run_grid ---------

def build_model_fn(cfg):
    """
    Build a Model instance from a single run config dict.
    """
    mcfg = cfg["model"]
    ocfg = cfg["optim"]
    rcfg = cfg.get("regularizer", {})
    cbcfg = cfg.get("callbacks", {})
    seed = int(cfg.get("seed", 0))

    hidden = int(mcfg.get("hidden_units", 16))
    dropout_p = float(mcfg.get("dropout", 0.0))
    act_name = str(mcfg.get("activation", "tanh")).lower()

    if act_name == "tanh":
        Act = Tanh
    elif act_name == "relu":
        Act = ReLU
    else:
        raise ValueError(f"Unknown activation: {act_name}")

    modules = [
        Dense(17, hidden, initializer=xavier_uniform),
        Act(),
    ]
    if dropout_p > 0:
        modules.append(Dropout(p=dropout_p, seed=seed))
    modules += [
        Dense(hidden, 1, initializer=xavier_uniform),
        Sigmoid(),
    ]

    # Regularizer (optional)
    reg = None
    l2 = float(rcfg.get("l2", 0.0))
    if l2 > 0:
        reg = L2(lam=l2)

    # Callbacks (EarlyStopping)
    callbacks = []
    if bool(cbcfg.get("early_stopping", True)):
        callbacks.append(EarlyStopping(
            monitor=str(cbcfg.get("monitor", "val_loss")),
            patience=int(cbcfg.get("patience", 50)),
            min_delta=float(cbcfg.get("min_delta", 0.0)),
            mode=str(cbcfg.get("mode", "auto")),
            restore_best_weights=bool(cbcfg.get("restore_best_weights", True)),
            verbose=int(cbcfg.get("verbose", 0)),
        ))

    model = Model(
        modules=modules,
        loss=BinaryCrossEntropy(),
        optimizer=SGD(
            lr=float(ocfg.get("lr", 0.1)),
        ),
        regularizer=reg,
        metrics=[Accuracy()],
        callbacks=callbacks,
    )
    return model


def load_data_fn(cfg):
    """
    Load and split train->train/val using your existing splitter.
    """
    dcfg = cfg["data"]
    split_cfg = cfg.get("split", {})
    seed = int(cfg.get("seed", 0))

    train_path = dcfg["train_path"]
    X, y = load_monk(train_path)

    val_size = float(split_cfg.get("val_size", 0.2))

    # Your split_train_val uses sklearn and accepts stratify=
    X_train, X_val, y_train, y_val = split_train_val(
        X, y,
        val_size=val_size,
        random_state=seed,
        stratify=y  # or y.ravel() depending on your splitter; y also usually works
    )
    return X_train, y_train, X_val, y_val


# --------- 2) Run a quick grid ---------

grid_yaml = "configs/monk1.yaml"          # <-- your YAML path
out_csv = "results/monk1_quick.csv"

res = run_grid(
    config_path=grid_yaml,
    build_model_fn=build_model_fn,
    load_data_fn=load_data_fn,
    out_csv_path=out_csv,
    seeds=[42],                            # quick test: single seed
    objective="val_loss",                  # or "val_Accuracy"
    objective_mode="auto",
    verbose=1,
)

print("\nBest score:", res["best_score"])
print("Best config:", res["best_config"])
print("Results written to:", out_csv)


# --------- 3) Retrain best config and evaluate on test ---------

best_cfg = res["best_config"]
seed = int(best_cfg.get("seed", 42))
np.random.seed(seed)

# Load train + split again
X_train_full, y_train_full = load_monk(best_cfg["data"]["train_path"])
X_test, y_test = load_monk(best_cfg["data"].get("test_path", "data/MONK/MONK1/monks-1.test"))

X_train, X_val, y_train, y_val = split_train_val(
    X_train_full, y_train_full,
    val_size=float(best_cfg.get("split", {}).get("val_size", 0.2)),
    random_state=seed,
    stratify=y_train_full
)

# Build + train with verbose output
model = build_model_fn(best_cfg)
trainer = Trainer(model, verbose=1)

history = trainer.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=int(best_cfg["training"].get("epochs", 500)),
    batch_size=int(best_cfg["training"].get("batch_size", 32)),
    shuffle=bool(best_cfg["training"].get("shuffle", True)),
    seed=seed,
)

print("\nTEST:", trainer.evaluate(X_test, y_test))


[grid] 1/8 run_id=96d8c4fc9ad4 seed=42
[grid] 2/8 run_id=bb9c09c1fcfd seed=42
[grid] 3/8 run_id=0f2c79947a24 seed=42
[grid] 4/8 run_id=57695e3ba9fc seed=42
[grid] 5/8 run_id=3c450d89e8a6 seed=42
[grid] 6/8 run_id=cf5ba6181b86 seed=42
[grid] 7/8 run_id=78397b01b499 seed=42
[grid] 8/8 run_id=4f553069a4b1 seed=42

Best score: 0.0813822545661452
Best config: {'callbacks': {'early_stopping': True, 'min_delta': 0.0, 'mode': 'auto', 'monitor': 'val_loss', 'patience': 40, 'restore_best_weights': True, 'verbose': 0}, 'data': {'monk': 1, 'train_path': 'data/MONK/MONK1/monks-1.train'}, 'model': {'activation': 'tanh', 'dropout': 0.0, 'hidden_units': 8}, 'optim': {'lr': 0.1, 'momentum': 0.0}, 'regularizer': {'l2': 0.0}, 'seed': 42, 'split': {'val_size': 0.2}, 'training': {'batch_size': 16, 'epochs': 800, 'include_reg_in_val': False, 'shuffle': True}}
Results written to: results/monk1_quick.csv

      _____     ___                 _ _ _ 
      \_   \   / __\__ ___   ____ _| | (_)
       / /\/  / /  